# 04 — Creator Scoring and Shortlisting

**Purpose:** Build composite scores that help partnerships teams quickly identify the right creators for different campaign objectives.

This notebook produces:
- **Creator Fit Score** — overall suitability for brand partnerships
- **Awareness Suitability Score** — optimized for reach and impressions
- **Engagement Suitability Score** — optimized for interaction quality
- **Shortlists** for each objective
- **Awareness vs Engagement quadrant chart** — the key visual for client decks

---

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from src.ingest import load_creators, load_posts
from src.clean import clean_creators, clean_posts
from src.features import add_post_features, build_creator_features
from src.scoring import (
    compute_creator_fit_score,
    compute_awareness_score,
    compute_engagement_score,
    generate_shortlist,
)
from src.utils import set_plot_style, COLORS, TIER_ORDER, TIER_COLORS, format_number

set_plot_style()

In [ ]:
creators = clean_creators(load_creators())
posts = add_post_features(clean_posts(load_posts()))
creators_enriched = build_creator_features(posts, creators)

# Compute all scores
scored = compute_creator_fit_score(creators_enriched)
scored = compute_awareness_score(scored)
scored = compute_engagement_score(scored)

print(f"Scored {len(scored)} creators")
print(f"Score columns: creator_fit_score, awareness_score, engagement_suitability_score")

## 1. Score Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, title, color in zip(
    axes,
    ['creator_fit_score', 'awareness_score', 'engagement_suitability_score'],
    ['Creator Fit Score', 'Awareness Score', 'Engagement Score'],
    [COLORS['primary'], COLORS['secondary'], COLORS['success']]
):
    ax.hist(scored[col], bins=25, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(scored[col].median(), color=COLORS['danger'], linestyle='--',
               label=f'Median: {scored[col].median():.1f}')
    ax.set_title(title)
    ax.set_xlabel('Score')
    ax.legend()

plt.tight_layout()
plt.show()

## 2. Awareness vs Engagement Quadrant

This is the key visual for a partnerships team: which creators are strong on reach, engagement, or both?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# Color by follower tier
tier_color_map = dict(zip(TIER_ORDER, TIER_COLORS))
colors = scored['follower_tier'].map(tier_color_map)

ax.scatter(
    scored['awareness_score'],
    scored['engagement_suitability_score'],
    c=colors, alpha=0.6, s=40, edgecolors='white', linewidth=0.5
)

# Quadrant lines at median
aw_med = scored['awareness_score'].median()
en_med = scored['engagement_suitability_score'].median()
ax.axvline(aw_med, color=COLORS['neutral'], linestyle='--', alpha=0.5)
ax.axhline(en_med, color=COLORS['neutral'], linestyle='--', alpha=0.5)

# Quadrant labels
ax.text(85, 90, 'STAR CREATORS\nHigh Reach + High Engagement', ha='center', fontsize=9, color=COLORS['success'], fontweight='bold')
ax.text(15, 90, 'ENGAGEMENT SPECIALISTS\nNiche + High Interaction', ha='center', fontsize=9, color=COLORS['secondary'], fontweight='bold')
ax.text(85, 10, 'AWARENESS DRIVERS\nBroad Reach + Lower ER', ha='center', fontsize=9, color=COLORS['primary'], fontweight='bold')
ax.text(15, 10, 'EMERGING CREATORS\nGrowing Audience', ha='center', fontsize=9, color=COLORS['neutral'], fontweight='bold')

ax.set_xlabel('Awareness Score')
ax.set_ylabel('Engagement Suitability Score')
ax.set_title('Creator Segmentation: Awareness vs Engagement')

# Legend
handles = [mpatches.Patch(color=c, label=t) for t, c in tier_color_map.items()]
ax.legend(handles=handles, title='Follower Tier', loc='lower right')

plt.tight_layout()
plt.savefig('../dashboard/awareness_vs_engagement_quadrant.png')
plt.show()

## 3. Top 30 Creators — Balanced Shortlist

In [ ]:
shortlist_balanced = generate_shortlist(scored, top_n=30, objective='balanced')

display_cols = [
    'rank', 'creator_id', 'category', 'follower_tier', 'followers',
    'avg_engagement_rate', 'sponsored_er', 'creator_fit_score',
    'awareness_score', 'engagement_suitability_score', 'recommendation'
]
shortlist_balanced[display_cols].head(15)

## 4. Awareness-Focused Shortlist

In [ ]:
shortlist_awareness = generate_shortlist(scored, top_n=15, objective='awareness')
shortlist_awareness[display_cols].head(10)

## 5. Engagement-Focused Shortlist

In [ ]:
shortlist_engagement = generate_shortlist(scored, top_n=15, objective='engagement')
shortlist_engagement[display_cols].head(10)

## 6. Score by Follower Tier and Category

In [ ]:
tier_scores = scored.groupby('follower_tier')[[
    'creator_fit_score', 'awareness_score', 'engagement_suitability_score'
]].mean().reindex(TIER_ORDER).round(1)

fig, ax = plt.subplots(figsize=(10, 5))
tier_scores.plot(kind='bar', ax=ax, color=[COLORS['primary'], COLORS['secondary'], COLORS['success']])
ax.set_title('Average Scores by Follower Tier')
ax.set_ylabel('Score')
ax.legend(title='Score Type')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../dashboard/scores_by_tier.png')
plt.show()

## 7. Creator Leaderboard — Top 20 Overall

In [ ]:
top20 = scored.nlargest(20, 'creator_fit_score')[[
    'creator_id', 'category', 'follower_tier', 'followers',
    'creator_fit_score', 'awareness_score', 'engagement_suitability_score',
    'consistency_score', 'sponsored_lift'
]].reset_index(drop=True)
top20.index = range(1, 21)
top20.index.name = 'Rank'

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(
    top20['creator_id'], top20['creator_fit_score'],
    color=COLORS['primary'], alpha=0.8
)
ax.set_xlabel('Creator Fit Score')
ax.set_title('Top 20 Creators by Fit Score')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../dashboard/creator_leaderboard.png')
plt.show()

## Summary

**Key findings for the partnerships team:**

1. **Creator Fit Scores** range from ~20 to ~85, with a clear top tier of well-rounded creators
2. **Micro and mid-tier creators** tend to score highest on the balanced fit metric — they offer the best combination of engagement quality, sponsored reliability, and consistency
3. **The quadrant chart reveals four distinct creator cohorts:**
   - Star Creators (high awareness + high engagement) — rare but ideal
   - Engagement Specialists — best for community-building and interaction-heavy campaigns
   - Awareness Drivers — best for reach and impression-focused campaigns
   - Emerging Creators — growing audiences, good for early partnerships
4. **Shortlists are objective-driven** — the awareness list skews larger creators while the engagement list skews smaller, more active creators

**Recommendation for partnerships teams:** Use the balanced shortlist as a starting point, then filter by campaign objective (awareness vs engagement) for client-specific recommendations.